# Notebook 07 — SOH Model Training

**Objective:** Train and evaluate 3 baseline regression models for SOH prediction.

**Models trained:**
1. Linear Regression (baseline / interpretable)
2. Random Forest Regressor (ensemble, captures non-linearity)
3. Gradient Boosting Regressor (sequential boosting)

**Train/test split:** Battery-level 70/30 (24 train, 10 test batteries, seed=42).
This prevents data leakage — no cycle from a test battery appears in training.

**Input features (10, leakage-free):**
`cycle, discharge_duration, energy_Wh, voltage_mean, voltage_min, voltage_std,`
`current_mean, temperature_mean, temperature_max, ambient_temp_C`

Note: `capacity` is EXCLUDED — it is the SOH target numerator.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

ROOT = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from soh import add_soh
from evaluation import compute_metrics, build_comparison_table, print_metrics_table

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11})
print('Libraries loaded')

## 1. Load Final Dataset

In [ ]:
final = pd.read_csv(os.path.join(ROOT, 'cleaned_data', 'final_dataset.csv'))
print(f'Shape: {final.shape}  |  Batteries: {final["battery_id"].nunique()}')
final.head()

## 2. Battery-Level Train/Test Split (70/30)

In [ ]:
rng = np.random.default_rng(42)
all_bats = sorted(final['battery_id'].unique())
n_test   = round(len(all_bats) * 0.30)
test_bats  = sorted(rng.choice(all_bats, size=n_test, replace=False).tolist())
train_bats = sorted([b for b in all_bats if b not in test_bats])

print(f'Train batteries ({len(train_bats)}): {train_bats}')
print(f'Test  batteries ({len(test_bats)}):  {test_bats}')

SOH_FEATURES = ['cycle', 'discharge_duration', 'energy_Wh',
                 'voltage_mean', 'voltage_min', 'voltage_std',
                 'current_mean', 'temperature_mean', 'temperature_max', 'ambient_temp_C']

soh_train = final[final['battery_id'].isin(train_bats)]
soh_test  = final[final['battery_id'].isin(test_bats)]

X_tr = soh_train[SOH_FEATURES].values;  y_tr = soh_train['soh'].values
X_te = soh_test[SOH_FEATURES].values;   y_te = soh_test['soh'].values

print(f'\nTrain: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'SOH train range: {y_tr.min():.1f}% – {y_tr.max():.1f}%')
print(f'SOH test  range: {y_te.min():.1f}% – {y_te.max():.1f}%')

## 3. Define Models (sklearn Pipelines with StandardScaler)

In [ ]:
models = {
    'Linear Regression': Pipeline([
        ('sc', StandardScaler()),
        ('m',  LinearRegression()),
    ]),
    'Random Forest': Pipeline([
        ('sc', StandardScaler()),
        ('m',  RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ]),
    'Gradient Boosting': Pipeline([
        ('sc', StandardScaler()),
        ('m',  GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ]),
}
print('Models defined:', list(models.keys()))

## 4. Train and Evaluate

In [ ]:
import time
results = []
fitted_models = {}

for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    tr_m = compute_metrics(y_tr, pipe.predict(X_tr), name)
    te_m = compute_metrics(y_te, pipe.predict(X_te), name)

    results.append({
        'model': name,
        'MAE_train': tr_m['MAE'],  'RMSE_train': tr_m['RMSE'],  'R2_train': tr_m['R2'],
        'MAE_test':  te_m['MAE'],  'RMSE_test':  te_m['RMSE'],  'R2_test':  te_m['R2'],
    })
    fitted_models[name] = pipe

    print(f'{name:<26}  '
          f'Train RMSE={tr_m["RMSE"]:.3f}%  R²={tr_m["R2"]:.3f}  |  '
          f'Test  RMSE={te_m["RMSE"]:.3f}%  R²={te_m["R2"]:.3f}  '
          f'({elapsed:.1f}s)')

## 5. Comparison Table

In [ ]:
table = build_comparison_table(results)
print_metrics_table(table, 'SOH Baseline Model Comparison')
table

## 6. Actual vs Predicted — All Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name in zip(axes, models.keys()):
    pipe = fitted_models[name]
    pred = pipe.predict(X_te)
    ax.scatter(y_te, pred, alpha=0.4, s=20, color='steelblue', edgecolors='k',
               linewidths=0.2)
    lim = [max(0, y_te.min()-5), y_te.max()+5]
    ax.plot(lim, lim, 'r--', linewidth=1.5)
    ax.set_xlabel('Actual SOH (%)')
    ax.set_ylabel('Predicted SOH (%)')
    rmse = next(r['RMSE_test'] for r in results if r['model'] == name)
    r2   = next(r['R2_test']   for r in results if r['model'] == name)
    ax.set_title(f'{name}\nRMSE={rmse:.3f}%  R²={r2:.3f}')
plt.tight_layout()
plt.show()

## 7. Feature Importances (Tree Models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name in zip(axes, ['Random Forest', 'Gradient Boosting']):
    pipe  = fitted_models[name]
    imps  = pipe.named_steps['m'].feature_importances_
    fi_df = pd.DataFrame({'feature': SOH_FEATURES, 'importance': imps})
    fi_df = fi_df.sort_values('importance', ascending=True)
    ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{name} — SOH Feature Importances')
plt.tight_layout()
plt.show()

## 8. Save Models

In [ ]:
os.makedirs(os.path.join(ROOT, 'models', 'soh'), exist_ok=True)
for name, pipe in fitted_models.items():
    stem = name.lower().replace(' ', '_')
    path = os.path.join(ROOT, 'models', 'soh', f'{stem}.pkl')
    joblib.dump(pipe, path)
    print(f'Saved: {path}')
print('\nAll SOH baseline models saved.')